# 01 · Catalogue snapshot (Bronze → Silver)

I pull every dataset listing for the health and social care publishers from the
data.gov.uk CKAN API and save the raw responses to Bronze under a UTC timestamp.
That raw snapshot is evidence in its own right. It records what the catalogue
said on the day I looked, which is the same habit the article argues for.

**Endpoint.** `https://www.data.gov.uk/api/action/*` 301-redirects to
`https://ckan.publishing.service.gov.uk/api/action/*`, so I call the CKAN host directly.

**Slugs.** I confirmed these against the `organization` facet of `package_search`
on 21 November 2026. See `config.PUBLISHERS`. NHS Digital and Public Health England
are predecessor bodies whose entries were never re-homed. I keep them as separate
rows and never fold them into NHS England or UKHSA.

In [1]:
import sys, pathlib
ROOT = pathlib.Path.cwd().resolve()
ROOT = ROOT.parent if ROOT.name == "notebooks" else ROOT
sys.path.insert(0, str(ROOT / "src"))
import pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
from hclr import config


In [2]:
from hclr import catalogue
# Re-running this cell takes a NEW dated snapshot; it never overwrites an old one.
RUN_SNAPSHOT = not list((config.BRONZE / "catalogue").glob("*Z"))
snap = catalogue.snapshot_publishers() if RUN_SNAPSHOT else catalogue.latest_snapshot()
print("snapshot:", snap.relative_to(ROOT))

snapshot: data/bronze/catalogue/2026-11-21T014501Z


In [3]:
import json
manifest = json.loads((snap / "manifest.json").read_text())
pd.DataFrame(manifest["files"])[["publisher", "page", "count_reported", "rows_in_page", "fetched_utc", "sha256"]]

,publisher,page,count_reported,rows_in_page,fetched_utc,sha256
0,nhs-england,0,37,37,2026-11-21T01:45:02.636297+00:00,46b3f0f81573f93909aa5592838d098ebba079958e8376...
1,department-of-health,0,164,100,2026-11-21T01:45:03.950391+00:00,d6cb993a7aa689997820527df4689c1468ed85e087a414...
2,department-of-health,1,164,64,2026-11-21T01:45:05.127262+00:00,3533aa253d0163b8190691be23f2ee37065e88079038d9...
3,nhs-business-services-authority,0,12,12,2026-11-21T01:45:06.274105+00:00,d3eb2d489345aeaa8393ca40c67164297b9578ade4c1cf...
4,uk-health-security-agency,0,1,1,2026-11-21T01:45:07.398655+00:00,b467b525bcbbe0dc6a3a9f7a7a4a8a85f321ed75d6274c...
5,nhs-digital,0,475,100,2026-11-21T01:45:08.580556+00:00,52c9477be57920ce995c234d7acb5030d0072e5bc0c792...
6,nhs-digital,1,475,100,2026-11-21T01:45:09.772259+00:00,bdc9806cf7811d02cdb8d25b49eaf1cbc79b65ba8ada58...
7,nhs-digital,2,475,100,2026-11-21T01:45:10.979123+00:00,13c1ba007695493f9cb1b82beebb6cc372aa2f624c7e93...
8,nhs-digital,3,475,100,2026-11-21T01:45:12.205525+00:00,4767e4334b9c5e2d97b6e7bcd6685af1e78eaa56e36fe7...
9,nhs-digital,4,475,75,2026-11-21T01:45:13.376230+00:00,fe4f2b4ca4d8ed2524afc61802ce6282d2a3d80dbf28bb...


In [4]:
datasets, resources = catalogue.flatten(snap)
datasets["snapshot_utc"] = resources["snapshot_utc"] = manifest["snapshot_utc"]
config.SILVER.mkdir(parents=True, exist_ok=True)
datasets.to_parquet(config.SILVER / "datasets.parquet", index=False)
resources.to_parquet(config.SILVER / "resources.parquet", index=False)
print(len(datasets), "datasets;", len(resources), "resource links;", resources["url"].nunique(), "distinct URLs")

823 datasets; 4305 resource links; 3111 distinct URLs


## What the snapshot holds, per publisher

`zero_resource_datasets` counts listings with no links left at all. Those are
catalogue entries that survive as metadata with nothing to follow.

In [5]:
per_pub = (datasets.groupby("publisher_slug")
    .agg(datasets=("dataset_id", "size"),
         zero_resource_datasets=("num_resources", lambda s: int((s == 0).sum())),
         harvest_marker=("harvest_marker", "sum"),
         oldest_modified=("metadata_modified", "min"),
         newest_modified=("metadata_modified", "max")))
per_pub = per_pub.join(resources.groupby("publisher_slug").agg(
    resource_links=("url", "size"), distinct_urls=("url", "nunique")))
per_pub

,datasets,zero_resource_datasets,harvest_marker,oldest_modified,newest_modified,resource_links,distinct_urls
publisher_slug,,,,,,,
department-of-health,164,35,0,2013-08-12 23:00:31.404269,2026-07-08 13:33:28.024457,1160,670
nhs-business-services-authority,12,0,0,2013-08-12 23:00:31.404269,2026-10-30 13:03:14.562757,198,193
nhs-digital,475,78,122,2013-08-12 23:00:31.404269,2026-07-08 13:33:29.348505,2489,2166
nhs-england,37,0,0,2014-05-09 22:37:59.806690,2026-11-11 15:50:48.890015,439,74
public-health-england,134,128,0,2013-08-30 16:30:46.518656,2026-07-08 13:33:29.461833,17,9
uk-health-security-agency,1,0,0,2022-11-18 13:16:18.183173,2022-11-18 13:16:18.183173,2,2


## When were these records last touched?

I check this because a cluster of `metadata_modified` dates in mid-2026 would be the
cleanup's own fingerprint. If so, "last updated" in this snapshot partly records when
the NDL edited a listing, not when the publisher updated the data.

In [6]:
(datasets.assign(ym=datasets["metadata_modified"].dt.to_period("M"))
   .query("ym >= '2026-01'")
   .pivot_table(index="ym", columns="publisher_slug", values="dataset_id", aggfunc="count", fill_value=0))

publisher_slug,department-of-health,nhs-business-services-authority,nhs-digital,nhs-england,public-health-england
ym,,,,,
2026-07,52,0,79,1,5
2026-08,0,7,0,0,0
2026-10,0,2,0,0,0
2026-11,0,0,0,1,0


In [7]:
from urllib.parse import urlsplit
resources["host"] = resources["url"].map(lambda u: urlsplit(u).netloc.lower())
(resources.groupby(["publisher_slug", "host"]).size().rename("links")
   .reset_index().sort_values(["publisher_slug", "links"], ascending=[True, False])
   .groupby("publisher_slug").head(5))

,publisher_slug,host,links
4,department-of-health,webarchive.nationalarchives.gov.uk,757
7,department-of-health,www.gov.uk,262
2,department-of-health,assets.publishing.service.gov.uk,94
5,department-of-health,www.apho.org.uk,12
10,department-of-health,www.nhsstaffsurveys.com,10
15,nhs-business-services-authority,www.nhsbsa.nhs.uk,180
14,nhs-business-services-authority,s3-eu-west-1.amazonaws.com,10
13,nhs-business-services-authority,opendata.nhsbsa.net,7
12,nhs-business-services-authority,cms.nhsbsa.nhs.uk,1
30,nhs-digital,www.hscic.gov.uk,1332
